In [1]:
from pathlib import Path
import os, subprocess

def get_project_root(max_up=6):
    try:
        root = subprocess.check_output(["git", "rev-parse", "--show-toplevel"], text=True).strip()
        if root:
            return Path(root)
    except Exception:
        pass
    p = Path.cwd()
    for _ in range(max_up):
        if (p / "data").exists() and (p / "code").exists():
            return p
        if (p / ".git").exists():
            return p
        p = p.parent
    return Path.cwd()

PROJECT_ROOT = get_project_root()
print("Project root:", PROJECT_ROOT)


Project root: /Users/liuq13/bhutan_climate_modeling


In [2]:

CATALOG_CSV          = PROJECT_ROOT / "data/basin_discharge/processed/stations_catalog.csv"
TEMPLATE_CSV         = PROJECT_ROOT / "data/modeling/targets/meta/station_to_basin_name.csv"

FINAL_MAPPING_CSV    = PROJECT_ROOT / "data/modeling/targets/meta/basin_station_choice.csv"
BASIN_LOOKUP_CSV     = PROJECT_ROOT / "data/boundaries/processed/basin_lookup.csv"

In [3]:
# --- Build template: station_name, use_for_model, basin_name ---

import pandas as pd
TEMPLATE_CSV.parent.mkdir(parents=True, exist_ok=True)

catalog = pd.read_csv(CATALOG_CSV)
if "station_name" not in catalog.columns:
    raise ValueError("stations_catalog.csv must include a 'station_name' column.")

template = catalog[["station_name"]].drop_duplicates().copy()

In [4]:
template

,station_name
0,Kurjey (Chamkharchhu)
1,Lungtenphu (Wangchhu)
2,Pangbang dangmechhu
3,Wangdi rapid (Punatshangchhu)
4,tingtibi (Mangdichhu)


In [7]:
template["use_for_model"] = [True, True, False, True, False]       
template["basin_name"]    = ["Mangdechhu", "Wangchhu", "", "Punatsangchhu", ""]

template.to_csv(TEMPLATE_CSV, index=False)
print("Wrote template:", TEMPLATE_CSV)
template

Wrote template: /Users/liuq13/bhutan_climate_modeling/data/modeling/targets/meta/station_to_basin_name.csv


,station_name,use_for_model,basin_name
0,Kurjey (Chamkharchhu),True,Mangdechhu
1,Lungtenphu (Wangchhu),True,Wangchhu
2,Pangbang dangmechhu,False,
3,Wangdi rapid (Punatshangchhu),True,Punatsangchhu
4,tingtibi (Mangdichhu),False,
